In [1]:
# ============================================================
# CS 499 Milestone Three
# Enhancement Two: Algorithms and Data Structures
# Animal Shelter Dashboard
# Samari Robinson Camacho
# ============================================================

# Standard-library import used to safely build breed-search patterns.
import re

# Dash imports
from dash import Dash, Input, Output, dash_table, dcc, html
import dash_leaflet as dl

# Data-processing import
import pandas as pd

# Import the enhanced CRUD module from Milestone Two.
from CRUD_Python_Module import CRUD


# ============================================================
# Application Configuration
# ============================================================

APP_TITLE = "SNHU CS-340 Animal Shelter Dashboard"
AUTHOR_NAME = "Samari Robinson Camacho"

DATABASE_HOST = "localhost"
DATABASE_PORT = 27017
DATABASE_NAME = "aac"
COLLECTION_NAME = "animals"

DEFAULT_MAP_CENTER = [30.75, -97.48]
DEFAULT_MAP_ZOOM = 10

TABLE_PAGE_SIZE = 10
MAP_WIDTH = "100%"
MAP_HEIGHT = "500px"


# ============================================================
# Algorithm and Data-Structure Configuration
# ============================================================

# A dictionary provides constant-time lookup of each rescue profile.
# It replaces repeated if/elif blocks with one reusable algorithm.
RESCUE_FILTERS = {
    "all": {
        "label": "All Animals",
        "breeds": [],
        "sex": None,
        "minimum_age_weeks": None,
        "maximum_age_weeks": None,
    },
    "water": {
        "label": "Water Rescue",
        "breeds": [
            "Labrador Retriever Mix",
            "Chesapeake Bay Retriever",
            "Newfoundland",
        ],
        "sex": "Intact Female",
        "minimum_age_weeks": 26,
        "maximum_age_weeks": 156,
    },
    "mountain": {
        "label": "Mountain or Wilderness Rescue",
        "breeds": [
            "German Shepherd",
            "Alaskan Malamute",
            "Old English Sheepdog",
            "Siberian Husky",
            "Rottweiler",
        ],
        "sex": "Intact Male",
        "minimum_age_weeks": 26,
        "maximum_age_weeks": 156,
    },
    "disaster": {
        "label": "Disaster or Individual Tracking",
        "breeds": [
            "Doberman Pinscher",
            "German Shepherd",
            "Golden Retriever",
            "Bloodhound",
            "Rottweiler",
        ],
        "sex": "Intact Male",
        "minimum_age_weeks": 20,
        "maximum_age_weeks": 300,
    },
}

# Candidate dataset names are stored in one dictionary. This allows the
# program to locate required fields even when column names vary slightly.
COLUMN_CANDIDATES = {
    "latitude": [
        "location_lat",
        "latitude",
        "lat",
    ],
    "longitude": [
        "location_long",
        "location_lon",
        "longitude",
        "long",
        "lon",
    ],
    "breed": [
        "breed",
    ],
    "name": [
        "name",
        "animal_name",
    ],
    "sex": [
        "sex_upon_outcome",
        "sex",
    ],
    "age_weeks": [
        "age_upon_outcome_in_weeks",
        "age_in_weeks",
        "age_weeks",
    ],
}


# ============================================================
# Database Connection
# ============================================================

def create_database_connection():
    """
    Connect to the local MongoDB Community Server.

    Returns:
        A configured CRUD object.

    Raises:
        ConnectionError: If MongoDB cannot be reached.
    """

    try:
        return CRUD(
            host=DATABASE_HOST,
            port=DATABASE_PORT,
            database_name=DATABASE_NAME,
            collection_name=COLLECTION_NAME,
        )

    except Exception as error:
        raise ConnectionError(
            f"Unable to connect to MongoDB: {error}"
        ) from error


# ============================================================
# Data Retrieval and Preparation
# ============================================================

def load_animal_records(database):
    """
    Retrieve animal records from MongoDB and prepare them for Dash.

    MongoDB ObjectId values are removed because they cannot be
    displayed directly by the Dash DataTable.

    Args:
        database: Connected CRUD object.

    Returns:
        A pandas DataFrame containing animal records.
    """

    try:
        records = database.read({})

        if records is None:
            records = []

        if not isinstance(records, list):
            records = list(records)

        dataframe = pd.DataFrame.from_records(records)

        if "_id" in dataframe.columns:
            dataframe = dataframe.drop(columns=["_id"])

        dataframe = dataframe.fillna("")

        return dataframe

    except Exception as error:
        print(f"Unable to retrieve animal records: {error}")
        return pd.DataFrame()


def build_column_lookup(dataframe, candidate_map):
    """
    Build a dictionary that maps logical field names to real columns.

    The dictionary gives later algorithms direct access to resolved
    columns instead of repeatedly scanning the DataFrame.

    Args:
        dataframe: DataFrame whose columns will be examined.
        candidate_map: Dictionary of logical names and possible columns.

    Returns:
        Dictionary mapping each logical name to a column name or None.
    """

    normalized_columns = {
        str(column).strip().lower(): column
        for column in dataframe.columns
    }

    lookup = {}

    for logical_name, possible_names in candidate_map.items():
        lookup[logical_name] = None

        for possible_name in possible_names:
            normalized_name = possible_name.strip().lower()

            if normalized_name in normalized_columns:
                lookup[logical_name] = normalized_columns[normalized_name]
                break

    return lookup


def create_table_columns(dataframe):
    """
    Create the Dash DataTable column configuration.

    Args:
        dataframe: Source DataFrame.

    Returns:
        List of DataTable column dictionaries.
    """

    return [
        {
            "name": column,
            "id": column,
            "deletable": False,
            "selectable": True,
        }
        for column in dataframe.columns
    ]


def filter_animals(dataframe, filter_key, column_lookup):
    """
    Apply the selected rescue-animal algorithm.

    The algorithm retrieves one rescue profile from RESCUE_FILTERS,
    builds Boolean masks for breed, sex, and age, combines the masks,
    and returns only records satisfying every requirement.

    Args:
        dataframe: Complete animal-record DataFrame.
        filter_key: Selected rescue-profile key.
        column_lookup: Resolved dataset column names.

    Returns:
        A tuple containing the filtered DataFrame and status message.
    """

    if dataframe.empty:
        return dataframe.copy(), "No animal records are available."

    # Dictionary lookup avoids a long repeated if/elif chain.
    profile = RESCUE_FILTERS.get(
        filter_key,
        RESCUE_FILTERS["all"],
    )

    if filter_key == "all":
        result = dataframe.copy().reset_index(drop=True)

        return (
            result,
            f"Showing all {len(result)} animal records.",
        )

    required_columns = {
        "breed": column_lookup.get("breed"),
        "sex": column_lookup.get("sex"),
        "age_weeks": column_lookup.get("age_weeks"),
    }

    missing_fields = [
        logical_name
        for logical_name, column_name in required_columns.items()
        if column_name is None
    ]

    if missing_fields:
        readable_fields = ", ".join(missing_fields)

        return (
            dataframe.iloc[0:0].copy(),
            "The selected rescue algorithm cannot run because the "
            f"dataset is missing: {readable_fields}.",
        )

    breed_column = required_columns["breed"]
    sex_column = required_columns["sex"]
    age_column = required_columns["age_weeks"]

    # Escape breed text before joining values into a regex pattern.
    breed_pattern = "|".join(
        re.escape(breed)
        for breed in profile["breeds"]
    )

    breed_mask = (
        dataframe[breed_column]
        .astype(str)
        .str.contains(
            breed_pattern,
            case=False,
            na=False,
            regex=True,
        )
    )

    sex_mask = (
        dataframe[sex_column]
        .astype(str)
        .str.strip()
        .str.casefold()
        .eq(profile["sex"].casefold())
    )

    numeric_age = pd.to_numeric(
        dataframe[age_column],
        errors="coerce",
    )

    age_mask = numeric_age.between(
        profile["minimum_age_weeks"],
        profile["maximum_age_weeks"],
        inclusive="both",
    )

    # Vectorized Boolean-mask combination performs the filtering without
    # repeatedly looping through each record in Python.
    combined_mask = breed_mask & sex_mask & age_mask

    filtered_dataframe = (
        dataframe.loc[combined_mask]
        .copy()
        .reset_index(drop=True)
    )

    # Apply a stable, predictable order to the filtered results.
    sort_columns = [
        column_name
        for column_name in [
            breed_column,
            column_lookup.get("name"),
        ]
        if column_name is not None
    ]

    if sort_columns:
        filtered_dataframe = filtered_dataframe.sort_values(
            by=sort_columns,
            kind="mergesort",
            ignore_index=True,
        )

    label = profile["label"]

    return (
        filtered_dataframe,
        f"{label}: {len(filtered_dataframe)} matching animal records.",
    )


# ============================================================
# Initialize Database and Data
# ============================================================

try:
    shelter = create_database_connection()
    df = load_animal_records(shelter)

except Exception as error:
    print(error)
    shelter = None
    df = pd.DataFrame()


# Build the lookup table once instead of repeatedly searching columns.
COLUMN_LOOKUP = build_column_lookup(
    df,
    COLUMN_CANDIDATES,
)

latitude_column = COLUMN_LOOKUP["latitude"]
longitude_column = COLUMN_LOOKUP["longitude"]
breed_column = COLUMN_LOOKUP["breed"]
name_column = COLUMN_LOOKUP["name"]


# ============================================================
# Dashboard Layout / View
# ============================================================

app = Dash(__name__)
app.title = APP_TITLE

app.layout = html.Div(
    children=[
        html.Div(
            children=[
                html.H1(
                    APP_TITLE,
                    style={
                        "textAlign": "center",
                        "marginBottom": "5px",
                    },
                ),
                html.H3(
                    f"Created by {AUTHOR_NAME}",
                    style={
                        "textAlign": "center",
                        "fontWeight": "normal",
                        "marginTop": "0",
                    },
                ),
            ]
        ),

        html.Hr(),

        html.H2("Rescue-Type Algorithm"),

        html.P(
            "Select a rescue category. The dashboard uses a "
            "dictionary-based profile and vectorized Boolean masks "
            "to identify suitable animals."
        ),

        dcc.RadioItems(
            id="rescue-filter",
            options=[
                {
                    "label": profile["label"],
                    "value": filter_key,
                }
                for filter_key, profile in RESCUE_FILTERS.items()
            ],
            value="all",
            inline=True,
            style={
                "marginBottom": "12px",
            },
            inputStyle={
                "marginRight": "5px",
                "marginLeft": "12px",
            },
        ),

        html.Div(
            id="filter-status",
            style={
                "fontWeight": "bold",
                "marginBottom": "15px",
            },
        ),

        html.H2("Animal Shelter Records"),

        html.P(
            "Use the table controls for additional sorting or filtering. "
            "Select one row to display the animal's location on the map."
        ),

        dash_table.DataTable(
            id="datatable-id",
            columns=create_table_columns(df),
            data=df.to_dict("records"),
            row_selectable="single",
            selected_rows=[0] if not df.empty else [],
            page_size=TABLE_PAGE_SIZE,
            sort_action="native",
            filter_action="native",
            style_table={
                "overflowX": "auto",
            },
            style_header={
                "fontWeight": "bold",
                "textAlign": "left",
                "backgroundColor": "#E8E8E8",
            },
            style_cell={
                "textAlign": "left",
                "minWidth": "120px",
                "width": "120px",
                "maxWidth": "180px",
                "whiteSpace": "normal",
                "height": "auto",
                "padding": "8px",
            },
        ),

        html.Br(),
        html.Hr(),

        html.H2("Selected Animal Location"),

        html.Div(
            id="map-message-id",
        ),

        html.Div(
            id="map-id",
            className="col s12 m6",
        ),
    ],

    style={
        "padding": "20px",
        "fontFamily": "Arial, sans-serif",
    },
)


# ============================================================
# Dashboard Controller
# ============================================================

@app.callback(
    [
        Output("datatable-id", "data"),
        Output("datatable-id", "selected_rows"),
        Output("filter-status", "children"),
    ],
    Input("rescue-filter", "value"),
)
def update_rescue_filter(selected_filter):
    """
    Update the table by running the selected rescue algorithm.

    Returns:
        Updated table records, reset row selection, and status text.
    """

    filtered_dataframe, message = filter_animals(
        df,
        selected_filter,
        COLUMN_LOOKUP,
    )

    selected_rows = [0] if not filtered_dataframe.empty else []

    return (
        filtered_dataframe.to_dict("records"),
        selected_rows,
        message,
    )


@app.callback(
    Output(
        "datatable-id",
        "style_data_conditional",
    ),
    Input(
        "datatable-id",
        "selected_columns",
    ),
)
def update_styles(selected_columns):
    """
    Highlight columns selected by the dashboard user.
    """

    if not selected_columns:
        return []

    return [
        {
            "if": {
                "column_id": column,
            },
            "backgroundColor": "#D2F3FF",
            "fontWeight": "bold",
        }
        for column in selected_columns
    ]


@app.callback(
    [
        Output(
            "map-id",
            "children",
        ),
        Output(
            "map-message-id",
            "children",
        ),
    ],
    [
        Input(
            "datatable-id",
            "derived_virtual_data",
        ),
        Input(
            "datatable-id",
            "derived_virtual_selected_rows",
        ),
    ],
)
def update_map(view_data, selected_rows):
    """
    Update the map using the selected animal record.

    Defensive validation prevents crashes when data, coordinates,
    columns, or row selections are missing or invalid.
    """

    if not view_data:
        return [], html.P(
            "No animal records are currently available."
        )

    filtered_dataframe = pd.DataFrame.from_records(
        view_data
    )

    if filtered_dataframe.empty:
        return [], html.P(
            "No records match the current filters."
        )

    if latitude_column is None or longitude_column is None:
        return [], html.P(
            "The dataset does not contain recognized "
            "latitude and longitude columns."
        )

    if selected_rows:
        row_index = selected_rows[0]
    else:
        row_index = 0

    if row_index < 0 or row_index >= len(filtered_dataframe):
        row_index = 0

    selected_record = filtered_dataframe.iloc[row_index]

    try:
        latitude = float(
            selected_record[latitude_column]
        )
        longitude = float(
            selected_record[longitude_column]
        )

    except (TypeError, ValueError, KeyError):
        return [], html.P(
            "The selected record does not contain valid coordinates."
        )

    animal_name = "Name unavailable"
    animal_breed = "Breed unavailable"

    if name_column and name_column in selected_record.index:
        name_value = str(
            selected_record[name_column]
        ).strip()

        if name_value:
            animal_name = name_value

    if breed_column and breed_column in selected_record.index:
        breed_value = str(
            selected_record[breed_column]
        ).strip()

        if breed_value:
            animal_breed = breed_value

    animal_map = dl.Map(
        style={
            "width": MAP_WIDTH,
            "height": MAP_HEIGHT,
        },
        center=[
            latitude,
            longitude,
        ],
        zoom=DEFAULT_MAP_ZOOM,
        children=[
            dl.TileLayer(
                id="base-layer-id",
            ),
            dl.Marker(
                position=[
                    latitude,
                    longitude,
                ],
                children=[
                    dl.Tooltip(
                        animal_breed,
                    ),
                    dl.Popup(
                        children=[
                            html.H3(
                                "Animal Information"
                            ),
                            html.P(
                                f"Name: {animal_name}"
                            ),
                            html.P(
                                f"Breed: {animal_breed}"
                            ),
                            html.P(
                                f"Coordinates: "
                                f"{latitude:.5f}, "
                                f"{longitude:.5f}"
                            ),
                        ]
                    ),
                ],
            ),
        ],
    )

    map_message = html.P(
        f"Displaying the shelter location for {animal_name}.",
        style={
            "fontWeight": "bold",
        },
    )

    return animal_map, map_message


# ============================================================
# Run the Dashboard
# ============================================================

if df.empty:
    print(
        "MongoDB is connected, but no animal records were loaded. "
        "Import the animal shelter dataset into the 'aac' database "
        "and the 'animals' collection."
    )

# Change the port to 8051 if port 8050 is already being used.
app.run(
    port=8050,
    debug=False,
)

Connected to MongoDB database 'aac' and collection 'animals'.
